# Fine-tune the Oyster Mushroom Cluster Model

Adapt the released YOLO26-L-seg model to another annotated mushroom dataset, check whether it improves on that dataset, and export a checkpoint for `monitor-your-data.ipynb`.

**Kaggle setup:** attach the [Oyster Mushroom Cluster Monitoring Models](https://www.kaggle.com/datasets/hnamedu/fungicirno-models) dataset and your annotated dataset, enable a GPU, then run all cells. All settings are in the next cell.

## 1. Configuration

Edit this cell only. Use a YOLO **instance-segmentation** dataset with one class: `mushroom_cluster`. Keep train, validation, and test images in separate splits. For time-lapse data, split by complete growing sessions/cameras rather than randomly splitting neighboring frames.

In [ ]:
from pathlib import Path

# =============================================================================
# CONFIG — change settings in this cell only
# =============================================================================

# Dataset and model
DATASET_ROOT = "/kaggle/input/your-annotated-mushroom-dataset"
DATA_YAML_PATH = None              # None = find one data.yaml under DATASET_ROOT; otherwise set its path
BASE_MODEL_PATH = None             # None = find selected_yolo26l_charisis.pt in attached Kaggle inputs
MODEL_DATASET_URL = "https://www.kaggle.com/datasets/hnamedu/fungicirno-models"
CLASS_NAME = "mushroom_cluster"   # This template expects one class with class ID 0

# Dataset checks
CHECK_IMAGE_FILES = True           # Try opening every image before training
CHECK_DUPLICATE_FILE_HASHES = False # Slower; checks identical image contents across splits
REQUIRE_TEST_SPLIT = False          # True = stop when no independent test split is present

# Training
EPOCHS = 100
IMAGE_SIZE = 1024
BATCH_SIZE = -1                    # -1 = Ultralytics auto batch sizing
WORKERS = 2
PATIENCE = 25
LEARNING_RATE = 0.0001             # Fine-tuning starting point; adjust using validation results
OPTIMIZER = "MuSGD"
DEVICE = "auto"                    # auto | cpu | 0
SEED = 42
DETERMINISTIC = True
AMP = True
CACHE_IMAGES = False

# Evaluation and previews
EVALUATE_TEST_SPLIT = True         # Test is only evaluated after training; never used for tuning
EVAL_IMAGE_SIZE = 1024
EVAL_BATCH_SIZE = 2
EVAL_CONFIDENCE = 0.001
EVAL_IOU = 0.70
PREVIEW_COUNT = 8                  # Number of test/validation images shown before and after fine-tuning

# Output
OUTPUT_DIR = "/kaggle/working/fine-tune-output"
RUN_NAME = "mushroom-cluster-finetune"
EXPORT_ZIP = True

## Dataset format

Expected layout:

```text
your-dataset/
├── data.yaml                 # optional; the notebook can make one
├── images/
│   ├── train/
│   ├── val/
│   └── test/                 # optional, but recommended
└── labels/
    ├── train/
    ├── val/
    └── test/
```

Each image needs a matching `.txt` file with YOLO segmentation polygons. Each non-empty line is:

```text
class_id x1 y1 x2 y2 x3 y3 ...
```

Coordinates must be normalized to 0–1 and each polygon needs at least three points. Use class ID `0` for mushroom clusters. The data.yaml must list exactly one class; its source name can differ because the notebook writes a normalized config using `CLASS_NAME`. An empty label file is valid for an image containing no mushrooms; a missing label file is treated as a dataset error.

If the dataset has several mushroom maturity labels, combine them into one `mushroom_cluster` class before using this template. This keeps the exported model compatible with the one-class monitoring workflow.

## 2. Load the dataset and check labels

This cell finds the dataset splits, checks image/label pairs and polygons, and writes a clean `data.yaml` for training. It stops before training if it finds missing labels, invalid polygons, unsupported classes, or images that cannot be opened.

In [ ]:
import os, re, sys, csv, json, math, time, shutil, hashlib, zipfile, subprocess, importlib.util, warnings
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

# Keep Kaggle's existing PyTorch/CUDA installation.
ULTRALYTICS_VERSION = "8.4.214"
try:
    current_version = version("ultralytics")
except PackageNotFoundError:
    current_version = None
if current_version != ULTRALYTICS_VERSION:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--no-deps",
        "--force-reinstall", f"ultralytics=={ULTRALYTICS_VERSION}"
    ])

missing_modules = [m for m in ["yaml", "cv2", "pandas", "matplotlib", "tqdm"] if importlib.util.find_spec(m) is None]
if missing_modules:
    packages = ["pyyaml" if m == "yaml" else "opencv-python-headless" if m == "cv2" else m for m in missing_modules]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

import yaml
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from tqdm.auto import tqdm
from ultralytics import YOLO

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda)
print("Ultralytics:", version("ultralytics"))
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "not available")

DATASET_ROOT_PATH = Path(DATASET_ROOT).expanduser().resolve()
if not DATASET_ROOT_PATH.is_dir():
    raise FileNotFoundError(f"Dataset folder not found: {DATASET_ROOT_PATH}")

OUTPUT_ROOT = Path(OUTPUT_DIR).expanduser().resolve()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
LABEL_SUFFIX = ".txt"


def find_data_yaml(root, explicit=None):
    if explicit:
        candidate = Path(explicit).expanduser().resolve()
        if not candidate.is_file():
            raise FileNotFoundError(f"DATA_YAML_PATH not found: {candidate}")
        return candidate
    candidates = sorted(root.rglob("data.yaml")) + sorted(root.rglob("data.yml"))
    candidates = [p for p in candidates if not any(part.startswith(".") for part in p.relative_to(root).parts)]
    if len(candidates) > 1:
        raise RuntimeError("More than one data.yaml was found. Set DATA_YAML_PATH explicitly: " + str(candidates[:10]))
    return candidates[0] if candidates else None


def as_names_list(names):
    if isinstance(names, dict):
        try:
            indexed = {int(k): str(v) for k, v in names.items()}
        except (TypeError, ValueError):
            raise ValueError("Class IDs in data.yaml 'names' must be integers starting at 0.")
        if sorted(indexed) != list(range(len(indexed))):
            raise ValueError("Class IDs in data.yaml 'names' must be contiguous and start at 0.")
        ordered = [indexed[i] for i in range(len(indexed))]
    elif isinstance(names, list):
        ordered = names
    else:
        raise ValueError("data.yaml must define class names under 'names'.")
    return [str(x) for x in ordered]


def resolve_yaml_split(value, yaml_dir):
    if value is None or value == "":
        return None
    values = value if isinstance(value, list) else [value]
    resolved = []
    for item in values:
        p = Path(str(item)).expanduser()
        if not p.is_absolute():
            p = yaml_dir / p
        p = p.resolve()
        if not p.exists():
            raise FileNotFoundError(f"Dataset split path from data.yaml does not exist: {p}")
        resolved.append(p)
    return resolved


def image_paths_from_entry(entry):
    if entry is None:
        return []
    found = []
    for path in entry:
        if path.is_dir():
            found.extend(p for p in path.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)
        elif path.suffix.lower() == ".txt":
            for line in path.read_text(encoding="utf-8").splitlines():
                line = line.strip()
                if not line:
                    continue
                p = Path(line)
                if not p.is_absolute():
                    p = path.parent / p
                if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS:
                    found.append(p.resolve())
                else:
                    raise FileNotFoundError(f"Image listed in split file is missing: {p}")
        elif path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
            found.append(path)
        else:
            raise ValueError(f"Unsupported split entry: {path}")
    return sorted(set(p.resolve() for p in found), key=lambda p: str(p).casefold())


def infer_label_path(image_path):
    parts = list(image_path.parts)
    image_positions = [i for i, part in enumerate(parts) if part.lower() == "images"]
    if not image_positions:
        raise ValueError(
            f"Cannot infer label path for {image_path}. Use the standard YOLO images/<split> and labels/<split> layout."
        )
    parts[image_positions[-1]] = "labels"
    return Path(*parts).with_suffix(".txt")


def find_standard_splits(root):
    candidates = {
        "train": [root / "images" / "train", root / "train" / "images"],
        "val": [root / "images" / "val", root / "images" / "valid", root / "val" / "images", root / "valid" / "images"],
        "test": [root / "images" / "test", root / "test" / "images"],
    }
    found = {}
    for split, paths in candidates.items():
        matches = [p for p in paths if p.is_dir()]
        if len(matches) > 1:
            raise RuntimeError(f"Multiple candidate {split} image folders found. Add a data.yaml to identify the intended split.")
        found[split] = [matches[0]] if matches else None
    if not found["train"] or not found["val"]:
        raise FileNotFoundError(
            "Could not find both train and validation folders. Provide data.yaml or use images/train and images/val."
        )
    return found


SOURCE_DATA_YAML = find_data_yaml(DATASET_ROOT_PATH, DATA_YAML_PATH)
if SOURCE_DATA_YAML:
    raw_yaml = yaml.safe_load(SOURCE_DATA_YAML.read_text(encoding="utf-8")) or {}
    source_names = as_names_list(raw_yaml.get("names"))
    if len(source_names) != 1:
        raise ValueError(
            f"This template trains the one-class mushroom_cluster model, but data.yaml lists {len(source_names)} classes: {source_names}. "
            "Merge mushroom maturity categories into class 0 before training."
        )
    split_base = Path(raw_yaml.get("path", SOURCE_DATA_YAML.parent))
    if not split_base.is_absolute():
        split_base = (SOURCE_DATA_YAML.parent / split_base).resolve()
    split_entries = {
        split: resolve_yaml_split(raw_yaml.get(split), split_base)
        for split in ["train", "val", "test"]
    }
    print("Dataset YAML:", SOURCE_DATA_YAML)
else:
    split_entries = find_standard_splits(DATASET_ROOT_PATH)
    source_names = [CLASS_NAME]
    print("No data.yaml found; using the standard images/labels folder layout.")

split_images = {split: image_paths_from_entry(entry) for split, entry in split_entries.items()}
for split in ["train", "val"]:
    if not split_images[split]:
        raise RuntimeError(f"The {split} split contains no images.")
if split_images["test"] and EVALUATE_TEST_SPLIT:
    print("Independent test split found; it will only be evaluated after training.")
elif REQUIRE_TEST_SPLIT:
    raise RuntimeError("No independent test split found, but REQUIRE_TEST_SPLIT=True.")
else:
    print("No test split found. Training/validation can proceed, but no held-out model comparison will be reported.")

all_resolved = {}
for split, images in split_images.items():
    if not images:
        continue
    resolved = set(images)
    for previous_split, previous_paths in all_resolved.items():
        overlap = resolved & previous_paths
        if overlap:
            raise RuntimeError(f"Image files overlap between {previous_split} and {split}: {list(overlap)[:5]}")
    all_resolved[split] = resolved


def validate_yolo_seg_split(split, images):
    object_count = 0
    empty_label_count = 0
    errors = []
    rows = []
    for image_path in tqdm(images, desc=f"Check {split}", unit="image"):
        label_path = infer_label_path(image_path)
        if not label_path.is_file():
            errors.append(f"Missing label: {label_path}")
            continue
        if CHECK_IMAGE_FILES:
            image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
            if image is None:
                errors.append(f"Unreadable image: {image_path}")
                continue
        lines = [line.strip() for line in label_path.read_text(encoding="utf-8").splitlines() if line.strip()]
        if not lines:
            empty_label_count += 1
        for line_number, line in enumerate(lines, start=1):
            fields = line.split()
            if len(fields) < 7 or (len(fields) - 1) % 2 != 0:
                errors.append(f"Invalid polygon (<3 points or malformed pairs>): {label_path}:{line_number}")
                continue
            try:
                class_id = int(fields[0])
                coords = np.asarray([float(v) for v in fields[1:]], dtype=np.float64)
            except ValueError:
                errors.append(f"Non-numeric label value: {label_path}:{line_number}")
                continue
            if class_id != 0:
                errors.append(f"Expected class ID 0, found {class_id}: {label_path}:{line_number}")
            if not np.isfinite(coords).all() or np.any(coords < 0) or np.any(coords > 1):
                errors.append(f"Polygon coordinates must be finite and normalized to [0, 1]: {label_path}:{line_number}")
            object_count += 1
        rows.append({"split": split, "image": str(image_path), "label": str(label_path), "objects": len(lines)})
    if errors:
        preview = "\n".join(errors[:30])
        raise RuntimeError(f"{split} split validation failed with {len(errors)} issue(s):\n{preview}")
    return {"images": len(images), "objects": object_count, "empty_labels": empty_label_count, "rows": rows}


split_stats = {}
manifest_rows = []
for split, images in split_images.items():
    if images:
        split_stats[split] = validate_yolo_seg_split(split, images)
        manifest_rows.extend(split_stats[split]["rows"])

DATASET_MANIFEST = pd.DataFrame(manifest_rows)
print("\nDATASET CHECK")
print(pd.DataFrame([
    {"split": s, "images": v["images"], "instances": v["objects"], "empty-label images": v["empty_labels"]}
    for s, v in split_stats.items()
]).to_string(index=False))

if CHECK_DUPLICATE_FILE_HASHES and len(split_images) > 1:
    hashes = {}
    for split, images in split_images.items():
        for image_path in tqdm(images, desc=f"Hash {split}"):
            digest = hashlib.sha256(image_path.read_bytes()).hexdigest()
            if digest in hashes and hashes[digest][0] != split:
                raise RuntimeError(f"Identical image content appears in both {hashes[digest][0]} and {split}: {image_path}")
            hashes[digest] = (split, image_path)

# Write a normalized YAML that always declares the one cluster class.
TRAINING_DATA_YAML = OUTPUT_ROOT / "mushroom_cluster_data.yaml"
normalized = {"path": str(DATASET_ROOT_PATH), "names": {0: CLASS_NAME}}
for split in ["train", "val", "test"]:
    entry = split_entries.get(split)
    if entry:
        normalized[split] = [str(p) for p in entry] if len(entry) > 1 else str(entry[0])
TRAINING_DATA_YAML.write_text(yaml.safe_dump(normalized, sort_keys=False), encoding="utf-8")
print("Training YAML:", TRAINING_DATA_YAML)

## 3. Check the annotations

A few labelled images are drawn below. Green outlines are the polygons from the label files. If these do not line up with the mushrooms, fix the annotations before training.

In [ ]:
from IPython.display import display


def draw_yolo_labels(image_path, label_path):
    image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if image is None:
        raise RuntimeError(f"Could not read {image_path}")
    h, w = image.shape[:2]
    for line in Path(label_path).read_text(encoding="utf-8").splitlines():
        fields = line.split()
        if len(fields) < 7:
            continue
        coords = np.asarray([float(v) for v in fields[1:]], dtype=np.float32).reshape(-1, 2)
        coords[:, 0] *= w
        coords[:, 1] *= h
        pts = np.round(coords).astype(np.int32).reshape(-1, 1, 2)
        cv2.polylines(image, [pts], isClosed=True, color=(0, 255, 0), thickness=max(2, round(min(h, w) / 500)))
    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)


rng = np.random.default_rng(SEED)
preview_images = split_images["train"]
chosen = rng.choice(len(preview_images), size=min(PREVIEW_COUNT, len(preview_images)), replace=False)
fig, axes = plt.subplots(len(chosen), 1, figsize=(12, max(4, 4 * len(chosen))))
if len(chosen) == 1:
    axes = [axes]
for ax, idx in zip(axes, chosen):
    image_path = preview_images[int(idx)]
    label_path = infer_label_path(image_path)
    ax.imshow(draw_yolo_labels(image_path, label_path))
    ax.set_title(f"{image_path.name} | {sum(1 for x in label_path.read_text().splitlines() if x.strip())} labelled clusters")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Load the starting model

The default is the released `selected_yolo26l_charisis.pt` checkpoint. Fine-tuning starts from these weights; it does not change the attached original file.

In [ ]:
def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def find_base_checkpoint(explicit_path=None):
    if explicit_path:
        candidate = Path(explicit_path).expanduser().resolve()
        if not candidate.is_file():
            raise FileNotFoundError(f"BASE_MODEL_PATH not found: {candidate}")
        return candidate
    target_name = "selected_yolo26l_charisis.pt"
    roots = [Path("/kaggle/input"), Path("/kaggle/working"), Path.cwd()]
    matches = set()
    for root in roots:
        if root.exists():
            for candidate in root.rglob(target_name):
                if candidate.is_file():
                    matches.add(candidate.resolve())
    matches = sorted(matches, key=str)
    if not matches:
        raise FileNotFoundError(
            f"Could not find {target_name}. Attach the model dataset linked in the first section, "
            "or set BASE_MODEL_PATH in the config cell."
        )
    hashes = {sha256_file(path) for path in matches}
    if len(hashes) > 1:
        raise RuntimeError(f"Different checkpoints with the same filename were found: {matches}. Set BASE_MODEL_PATH explicitly.")
    return matches[0]


BASE_CHECKPOINT = find_base_checkpoint(BASE_MODEL_PATH)
BASE_CHECKPOINT_SHA256 = sha256_file(BASE_CHECKPOINT)
base_model = YOLO(str(BASE_CHECKPOINT))
if getattr(base_model, "task", None) != "segment":
    raise ValueError(f"The starting checkpoint must be a YOLO segmentation model; found task={getattr(base_model, 'task', None)}")
print("Starting checkpoint:", BASE_CHECKPOINT)
print("SHA-256:", BASE_CHECKPOINT_SHA256)
print("Model task:", base_model.task)
print("Model classes:", base_model.names)

## 5. Fine-tune

Training uses the training split. The validation split is used for epoch selection and early stopping. The test split is checked for file/label integrity earlier, but it is not used for fitting or checkpoint selection.

In [ ]:
def resolve_device(value):
    if str(value).lower() == "auto":
        return 0 if torch.cuda.is_available() else "cpu"
    return value


TRAIN_DEVICE = resolve_device(DEVICE)
run_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_name = f"{RUN_NAME}_{run_stamp}"

train_model = YOLO(str(BASE_CHECKPOINT))
train_results = train_model.train(
    data=str(TRAINING_DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    workers=WORKERS,
    patience=PATIENCE,
    optimizer=OPTIMIZER,
    lr0=LEARNING_RATE,
    device=TRAIN_DEVICE,
    seed=SEED,
    deterministic=DETERMINISTIC,
    amp=AMP,
    cache=CACHE_IMAGES,
    pretrained=True,
    val=True,
    plots=True,
    save=True,
    project=str(OUTPUT_ROOT / "training_runs"),
    name=run_name,
    exist_ok=False,
    verbose=True,
)

TRAIN_RUN_DIR = Path(train_model.trainer.save_dir).resolve()
BEST_CHECKPOINT = TRAIN_RUN_DIR / "weights" / "best.pt"
LAST_CHECKPOINT = TRAIN_RUN_DIR / "weights" / "last.pt"
if not BEST_CHECKPOINT.is_file():
    raise FileNotFoundError(f"Training finished but best.pt was not found: {BEST_CHECKPOINT}")
print("\nTraining run:", TRAIN_RUN_DIR)
print("Best checkpoint:", BEST_CHECKPOINT)
print("Last checkpoint:", LAST_CHECKPOINT if LAST_CHECKPOINT.exists() else "not saved")

## 6. Review training

Check that training and validation losses behave sensibly. A large train/validation gap can indicate overfitting. The best checkpoint is selected by the training framework's validation fitness; do not choose an epoch using the test split.

In [ ]:
results_csv = TRAIN_RUN_DIR / "results.csv"
if results_csv.is_file():
    history = pd.read_csv(results_csv)
    history.columns = [c.strip() for c in history.columns]
    display(history.tail(10))
    numeric_columns = [c for c in history.columns if c != "epoch" and pd.api.types.is_numeric_dtype(history[c])]
    preferred = [c for c in numeric_columns if any(k in c.lower() for k in ["loss", "map50", "map50-95", "precision", "recall"])]
    for column in preferred:
        plt.figure(figsize=(7, 3))
        plt.plot(history["epoch"], history[column])
        plt.xlabel("Epoch")
        plt.ylabel(column)
        plt.title(column)
        plt.grid(alpha=0.25)
        plt.show()
else:
    print("Training results.csv was not found in", TRAIN_RUN_DIR)

## 7. Compare the original and fine-tuned models

If you supplied an independent `test` split, both checkpoints are evaluated on exactly that split with the same image size, batch size, confidence threshold, and IoU setting. The test split is only used here, after training is complete.

If no test split exists, this section reports that the comparison was skipped. It does not substitute the validation split and call it a test result. If you use test metrics or previews to change training settings, that split has become part of development; keep another untouched split for a final evaluation.

In [ ]:
def metric_value(metrics, group, field):
    obj = getattr(metrics, group, None)
    value = getattr(obj, field, None) if obj is not None else None
    try:
        return float(value)
    except (TypeError, ValueError):
        return np.nan


def collect_metrics(label, metrics):
    row = {"model": label}
    for prefix, group in [("box", "box"), ("mask", "seg")]:
        for field, display_name in [("mp", "precision"), ("mr", "recall"), ("map50", "mAP50"), ("map75", "mAP75"), ("map", "mAP50-95")]:
            row[f"{prefix}_{display_name}"] = metric_value(metrics, group, field)
    for prefix in ["box", "mask"]:
        p_value, r_value = row[f"{prefix}_precision"], row[f"{prefix}_recall"]
        row[f"{prefix}_f1"] = (2 * p_value * r_value / (p_value + r_value)) if np.isfinite(p_value) and np.isfinite(r_value) and (p_value + r_value) > 0 else np.nan
    return row


comparison_rows = []
TEST_IMAGES = split_images.get("test", [])
TEST_EVALUATED = bool(TEST_IMAGES) and EVALUATE_TEST_SPLIT

if TEST_EVALUATED:
    common_val_args = {
        "data": str(TRAINING_DATA_YAML),
        "split": "test",
        "imgsz": EVAL_IMAGE_SIZE,
        "batch": EVAL_BATCH_SIZE,
        "device": TRAIN_DEVICE,
        "conf": EVAL_CONFIDENCE,
        "iou": EVAL_IOU,
        "plots": True,
        "verbose": True,
    }
    print("Evaluating original checkpoint on held-out test split...")
    original_eval_model = YOLO(str(BASE_CHECKPOINT))
    original_test_metrics = original_eval_model.val(**common_val_args, project=str(OUTPUT_ROOT / "evaluation"), name="original_test", exist_ok=True)
    comparison_rows.append(collect_metrics("original", original_test_metrics))

    print("Evaluating fine-tuned checkpoint on the same held-out test split...")
    finetuned_eval_model = YOLO(str(BEST_CHECKPOINT))
    finetuned_test_metrics = finetuned_eval_model.val(**common_val_args, project=str(OUTPUT_ROOT / "evaluation"), name="finetuned_test", exist_ok=True)
    comparison_rows.append(collect_metrics("fine_tuned", finetuned_test_metrics))

    comparison = pd.DataFrame(comparison_rows).set_index("model")
    comparison.loc["change (fine-tuned - original)"] = comparison.loc["fine_tuned"] - comparison.loc["original"]
    display(comparison)
    comparison.to_csv(OUTPUT_ROOT / "test_model_comparison.csv")
else:
    comparison = pd.DataFrame()
    print("Test comparison skipped: provide an independent test split and set EVALUATE_TEST_SPLIT=True.")

## 8. Look at predictions

The same few held-out images are shown for the original and fine-tuned checkpoints. Green outlines are the ground-truth polygons; predicted masks are shown by YOLO. Use these examples to spot missed clusters, extra detections, poor boundaries, and changes that the aggregate metrics may hide.

In [ ]:
preview_source = TEST_IMAGES if TEST_EVALUATED else split_images["val"]
preview_source_name = "test" if TEST_EVALUATED else "validation"
preview_rng = np.random.default_rng(SEED)
selected_indices = preview_rng.choice(len(preview_source), size=min(PREVIEW_COUNT, len(preview_source)), replace=False)
selected_images = [preview_source[int(i)] for i in selected_indices]

original_preview_model = YOLO(str(BASE_CHECKPOINT))
finetuned_preview_model = YOLO(str(BEST_CHECKPOINT))


def draw_ground_truth(image_path):
    image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    h, w = image.shape[:2]
    label_path = infer_label_path(image_path)
    for line in label_path.read_text(encoding="utf-8").splitlines():
        fields = line.split()
        if len(fields) < 7:
            continue
        points = np.asarray([float(v) for v in fields[1:]], dtype=np.float32).reshape(-1, 2)
        points[:, 0] *= w
        points[:, 1] *= h
        cv2.polylines(image, [np.round(points).astype(np.int32).reshape(-1, 1, 2)], True, (0, 255, 0), 2)
    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)


for image_path in selected_images:
    original_pred = original_preview_model.predict(str(image_path), imgsz=EVAL_IMAGE_SIZE, conf=EVAL_CONFIDENCE, device=TRAIN_DEVICE, verbose=False)[0]
    fine_pred = finetuned_preview_model.predict(str(image_path), imgsz=EVAL_IMAGE_SIZE, conf=EVAL_CONFIDENCE, device=TRAIN_DEVICE, verbose=False)[0]
    gt_image = draw_ground_truth(image_path)
    original_plot = cv2.cvtColor(original_pred.plot(), cv2.COLOR_BGR2RGB)
    fine_plot = cv2.cvtColor(fine_pred.plot(), cv2.COLOR_BGR2RGB)
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(gt_image); axes[0].set_title("Ground truth"); axes[0].axis("off")
    axes[1].imshow(original_plot); axes[1].set_title("Original checkpoint"); axes[1].axis("off")
    axes[2].imshow(fine_plot); axes[2].set_title("Fine-tuned checkpoint"); axes[2].axis("off")
    fig.suptitle(f"{preview_source_name}: {image_path.name}")
    plt.tight_layout(); plt.show()

## 9. Export the model

The exported `best.pt` is the checkpoint to attach to `monitor-your-data.ipynb`; point its `YOLO_WEIGHTS_PATH` setting to this file. The original checkpoint is left untouched. Keep the test comparison and training plots with the model so you can see which dataset and settings produced it.

In [ ]:
EXPORT_DIR = OUTPUT_ROOT / "exported_model"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

export_best = EXPORT_DIR / "best.pt"
shutil.copy2(BEST_CHECKPOINT, export_best)
if LAST_CHECKPOINT.is_file():
    shutil.copy2(LAST_CHECKPOINT, EXPORT_DIR / "last.pt")

# Include training curves, weights metadata, evaluation files, and the exact data config.
for filename in ["results.csv", "args.yaml", "results.png", "confusion_matrix.png", "BoxPR_curve.png", "MaskPR_curve.png"]:
    candidate = TRAIN_RUN_DIR / filename
    if candidate.is_file():
        shutil.copy2(candidate, EXPORT_DIR / candidate.name)
shutil.copy2(TRAINING_DATA_YAML, EXPORT_DIR / "training_data.yaml")

run_record = {
    "created_at": datetime.now().astimezone().isoformat(),
    "base_checkpoint": str(BASE_CHECKPOINT),
    "base_checkpoint_sha256": BASE_CHECKPOINT_SHA256,
    "exported_checkpoint": str(export_best),
    "exported_checkpoint_sha256": sha256_file(export_best),
    "dataset_root": str(DATASET_ROOT),
    "source_data_yaml": str(SOURCE_DATA_YAML) if SOURCE_DATA_YAML else None,
    "class_name": CLASS_NAME,
    "split_counts": {name: {"images": values["images"], "instances": values["objects"], "empty_labels": values["empty_labels"]} for name, values in split_stats.items()},
    "training": {
        "epochs": EPOCHS, "image_size": IMAGE_SIZE, "batch_size": BATCH_SIZE, "workers": WORKERS,
        "patience": PATIENCE, "learning_rate": LEARNING_RATE, "optimizer": OPTIMIZER,
        "device": str(TRAIN_DEVICE), "seed": SEED, "deterministic": DETERMINISTIC,
    },
    "test_evaluated": TEST_EVALUATED,
}
(EXPORT_DIR / "run_info.json").write_text(json.dumps(run_record, indent=2), encoding="utf-8")

if TEST_EVALUATED:
    shutil.copy2(OUTPUT_ROOT / "test_model_comparison.csv", EXPORT_DIR / "test_model_comparison.csv")

zip_path = OUTPUT_ROOT / "fine_tuned_model_bundle.zip"
if EXPORT_ZIP:
    if zip_path.exists(): zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=4) as archive:
        for path in EXPORT_DIR.rglob("*"):
            if path.is_file(): archive.write(path, path.relative_to(OUTPUT_ROOT))

print("\nFINE-TUNING COMPLETE")
print("Best checkpoint:", export_best)
print("SHA-256:", sha256_file(export_best))
print("Model bundle:", zip_path if EXPORT_ZIP else "zip export disabled")
print("Attach best.pt to monitor-your-data.ipynb and set YOLO_WEIGHTS_PATH to its path.")

## Notes

- This notebook expects polygon segmentation labels, not bounding-box-only detection labels.
- It trains one class (`mushroom_cluster`). Convert multiple mushroom maturity classes into class ID `0` before training.
- Keep the test split separate. Training and checkpoint selection use train/validation only; test metrics are calculated after training.
- Fine-tuning on one farm can improve that environment but does not guarantee better performance on other farms. Keep the original checkpoint and compare both on representative held-out images.